# Chapter 11: Network Defense and Architecture

> "Defense in depth is not a product you buy; it is a strategy you design." Adapted security principle

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Explain defense in depth and layered network architecture.
2. Compare firewall types and placement.
3. Describe segmentation, DMZ design, and zero-trust principles.
4. Explain how VPNs and proxies fit a defensive architecture.
5. Reason about logging and monitoring as foundational controls.

## Key Terms

- **DMZ**: Demilitarized Zone, a buffer network for public-facing services.
- **Zero Trust**: An architecture that never implicitly trusts based on network location.
- **NGFW**: Next-Generation Firewall.
- **Segmentation**: Dividing a network into isolated zones.
- **Egress filtering**: Controlling outbound traffic.

---

## 11.1 Defense in Depth

No single control is sufficient, so sound architecture layers controls so that the failure of one does
not lead to compromise. The classic image is a series of concentric defenses, each independent, so that
an attacker must defeat many to reach the asset. Defense in depth also buys time and generates the
signals that detection relies upon.

## 11.2 Firewalls and Placement

Firewalls enforce traffic policy and have evolved from stateless packet filters to stateful inspection
and to next-generation firewalls that understand applications and users. Placement matters as much as
capability: a firewall at the perimeter, between internal zones, and around sensitive enclaves enforces
different policies appropriate to each boundary. Egress filtering, restricting what may leave, is often
neglected yet valuable for containing compromise.

## 11.3 Segmentation and the DMZ

Segmentation divides a network so that a foothold in one zone does not grant access to all others. A
demilitarized zone isolates public-facing servers from the internal network, so that compromising a web
server does not directly expose internal databases. Microsegmentation extends this idea down to
individual workloads, sharply limiting lateral movement.

## 11.4 Zero Trust

Zero trust rejects the assumption that being inside the network implies trust. Instead, every request is
authenticated, authorized, and encrypted based on identity and device posture rather than location.
This model responds to the reality that perimeters are porous and that attackers, once inside, should
not find a soft interior to roam freely.

## 11.5 Why This Matters

Architecture decisions are difficult and expensive to reverse, so getting segmentation, trust
boundaries, and monitoring right early pays dividends for years. Most large breaches are made worse by
flat networks that allow easy lateral movement after an initial foothold.

## 11.6 News in Focus

The 2013 breach of a major U.S. retailer, in which attackers entered through a third-party vendor
connection and reached point-of-sale systems, is a frequently cited example of how inadequate
segmentation lets a limited initial compromise escalate into a massive data breach.

## 11.7 Worked Example: Evaluating Segmentation

The code models network zones and connectivity rules and reports whether a sensitive zone is reachable
from the internet, illustrating how architecture can be checked rather than assumed.


In [1]:
zones = ["internet", "dmz", "internal", "database"]

# allowed[a] = set of zones reachable directly from a
allowed = {
    "internet": {"dmz"},
    "dmz":      {"internal"},
    "internal": {"database"},
    "database": set(),
}

def reachable(start):
    seen, stack = set(), [start]
    while stack:
        z = stack.pop()
        for nxt in allowed.get(z, set()):
            if nxt not in seen:
                seen.add(nxt); stack.append(nxt)
    return seen

from_internet = reachable("internet")
print("Reachable from internet:", sorted(from_internet))
print("Database directly internet-reachable:", "database" in allowed["internet"])
print("Database reachable via chained hops:", "database" in from_internet)
print("\nLesson: even with no direct rule, transitive paths can expose sensitive zones.")
print("Mitigation: require authentication and policy at each hop, not just the perimeter.")


Reachable from internet: ['database', 'dmz', 'internal']
Database directly internet-reachable: False
Database reachable via chained hops: True

Lesson: even with no direct rule, transitive paths can expose sensitive zones.
Mitigation: require authentication and policy at each hop, not just the perimeter.


## 11.8 Review Questions (MCQ)

**Q1.** A DMZ is designed to:
A. Encrypt traffic  B. Isolate public-facing servers  C. Store backups  D. Replace a firewall

**Q2.** Zero trust bases access decisions primarily on:
A. Network location  B. Identity and device posture  C. IP reputation  D. Port number

**Q3.** Restricting outbound traffic is called:
A. Ingress filtering  B. Egress filtering  C. NAT  D. Tunneling

*Answers: Q1 B, Q2 B, Q3 B.*

## 11.9 Lab Assignment

Design a network diagram for a small business with a public web server, internal workstations, and a
database. Define the zones, the allowed flows between them, and the egress policy. Explain how your
design limits lateral movement after a single compromised workstation.

## References

```{bibliography}
:filter: docname in docnames
```
